### Classificação de Sentimentos com RNN do Scikit-Learn

Importando as bibliotecas:

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

Gerando a amostra de dados a ser processada:

In [8]:
data = [
    ("O filme é maravilhoso! Adorei.", 1),
    ("O enredo estava confuso, e a atuação não foi boa.", 0),
    ("Este restaurante tem a melhor comida da cidade.", 1),
    ("O serviço foi terrível, e o lugar estava sujo.", 0)
]

# Converter dados para um DataFrame
df = pd.DataFrame(data, columns=['text', 'sentiment'])


Tokenizar e ajustar os dados:

In [9]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['text'])
sequences = tokenizer.texts_to_sequences(df['text'])
word_index = tokenizer.word_index
max_length = max(len(sequence) for sequence in sequences)
data_padded = pad_sequences(sequences, maxlen=max_length)

Codificação para rótulos:

In [10]:
encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(df['sentiment'])
labels_one_hot = to_categorical(labels_encoded)

Divisão da amostra em dados de teste e treinamento:

In [11]:
X_train, X_test, y_train, y_test = train_test_split(data_padded, labels_one_hot, test_size=0.25, random_state=42)

Compilação da RNN:

In [12]:
model = Sequential()
model.add(Embedding(input_dim=len(word_index) + 1, output_dim=64, input_length=max_length))
model.add(LSTM(64))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(2, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

/home/bruno/.venvs/ml/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
E0000 00:00:1781835733.361110  129982 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Treino do Modelo:

In [13]:
model.fit(X_train, y_train, epochs=10, batch_size=2)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 1.0000 - loss: 0.6890
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 1.0000 - loss: 0.6798
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.6765
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 1.0000 - loss: 0.6762
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.6541
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.6504
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6667 - loss: 0.6492
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.6401
Epoch 9/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.6134
Epoch 10/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.6526


Avaliação do Modelo:

In [14]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.0000e+00 - loss: 0.6947
Test Accuracy: 0.0


Fazendo uma previsões com o modelo:

In [15]:
# Testar uma frase
test_sentence = "O serviço não foi bom."

# Tokenizar e ajustar a frase de testes
test_sequence = tokenizer.texts_to_sequences([test_sentence])
test_padded = pad_sequences(test_sequence, maxlen=max_length)

# Fazer a classificação
prediction = model.predict(test_padded)[0]
sentiment_label = "positivo" if prediction[1] > prediction[0] else "negativo"
confidence = max(prediction) * 100

print(f"Frase de Teste: '{test_sentence}'")
print(f"Sentimento: {sentiment_label}")
print(f"Confiança: {confidence:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
Frase de Teste: 'O serviço não foi bom.'
Sentimento: positivo
Confiança: 56.20%


In [16]:
# Testar uma frase
test_sentence = "A comida estava deliciosa, e o atendimento foi ótimo."

# Tokenizar e ajustar a frase de testes
test_sequence = tokenizer.texts_to_sequences([test_sentence])
test_padded = pad_sequences(test_sequence, maxlen=max_length)

# Fazer a classificação
prediction = model.predict(test_padded)[0]
sentiment_label = "positivo" if prediction[1] > prediction[0] else "negativo"
confidence = max(prediction) * 100

print(f"Frase de Teste: '{test_sentence}'")
print(f"Sentimento: {sentiment_label}")
print(f"Confiança: {confidence:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Frase de Teste: 'A comida estava deliciosa, e o atendimento foi ótimo.'
Sentimento: positivo
Confiança: 54.41%
